# Objective 2 · Session-level smartphone behaviour

At the start of a smartphone session, was the user opening sessions unusually often, making unusually frequent short visits to apps, or doing both? This notebook reproduces the rule-based analysis in `Research_Project_Objective_2.ipynb`. Each session is compared with the same user's earlier behaviour.

The workflow moves from **event records** to **sessions**, reconstructs continuous stretches of app use called **bouts**, and then measures recent activity at each session start. The final table, `decision_features`, contains one row per user–session pair and the label `engagement_pattern_p75`. The intermediate `sessions` and `bout_audit` tables let you trace a label back to the events that contributed to it.

## 1. Load the Objective 1 event stream

Run the cells in order with a Python 3 Jupyter kernel, pandas, NumPy and openpyxl. The input is the prepared event stream in `Datasets/lsapp_objective1.xlsx`. The setup finds the file when Jupyter starts in the project root or the `Notebooks` directory. If your workbook is elsewhere, set `data_path` to its location.

An **episode** has a recorded start, end and duration for one app. An **interaction** is a point observation. Both carry the user ID, session ID and event order needed to put them back into context.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

session_keys = ["user_id", "session_id"]
order_columns = ["user_id", "session_start", "session_id"]
short_limit_seconds = 15

project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
data_path = project_root / "Datasets" / "lsapp_objective1.xlsx"

df = pd.read_excel(data_path, engine="openpyxl")
required = session_keys + [
    "app_name", "event_class", "timestamp", "start_time", "end_time",
    "duration_seconds", "session_event_number",
]
missing = set(required).difference(df.columns)
if missing:
    raise ValueError(f"Missing event columns in {data_path.name}: {sorted(missing)}")

for column in ["timestamp", "start_time", "end_time"]:
    df[column] = pd.to_datetime(df[column], errors="raise")

n_users = df["user_id"].nunique()
n_sessions = df[session_keys].drop_duplicates().shape[0]
print(f"{data_path.name} | {len(df):,} events | {n_users:,} users | {n_sessions:,} sessions")
display(df.head())

: 

## 2. Audit what the log measures

First check the event classes and where a duration is available. Then look at zero-duration episodes. When timestamps are recorded to whole seconds, equal start and end times leave a brief visit indistinguishable from an instantaneous record. Keep these episodes in counts and give them zero recorded seconds. The short-use measure requires a positive duration below 15 seconds.

In [2]:
df.info()
is_episode = df["event_class"].eq("episode")
is_interaction = df["event_class"].eq("interaction")
assert (is_episode | is_interaction).all()

duration_coverage = pd.crosstab(
    df["event_class"], df["duration_seconds"].notna(), margins=True,
    colnames=["Duration recorded"],
)
display(duration_coverage)
print("Event timestamps with fractional seconds:", df["timestamp"].dt.microsecond.ne(0).sum())

zero_episodes = df.loc[is_episode & df["duration_seconds"].eq(0)]
print(f"Zero-duration episodes: {len(zero_episodes):,} of {is_episode.sum():,} "
      f"({len(zero_episodes) / is_episode.sum():.2%})")
print("With identical start and end:", zero_episodes["start_time"].eq(zero_episodes["end_time"]).sum())
display(zero_episodes[session_keys + ["app_name", "start_time", "end_time", "duration_seconds"]].head())
display(zero_episodes["app_name"].value_counts().head(10).rename("zero_duration_episodes").to_frame())

<class 'pandas.DataFrame'>
RangeIndex: 899349 entries, 0 to 899348
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   user_id               899349 non-null  int64         
 1   session_id            899349 non-null  int64         
 2   episode_number        585520 non-null  float64       
 3   app_name              899349 non-null  str           
 4   timestamp             899349 non-null  datetime64[us]
 5   event_class           899349 non-null  str           
 6   start_time            585520 non-null  datetime64[us]
 7   end_time              585520 non-null  datetime64[us]
 8   duration_seconds      585520 non-null  float64       
 9   session_event_number  899349 non-null  int64         
dtypes: datetime64[us](3), float64(2), int64(3), str(2)
memory usage: 68.6 MB


Duration recorded,False,True,All
event_class,,,
episode,0,585520,585520
interaction,313829,0,313829
All,313829,585520,899349


Event timestamps with fractional seconds: 0
Zero-duration episodes: 55,073 of 585,520 (9.41%)
With identical start and end: 55073


,user_id,session_id,app_name,start_time,end_time,duration_seconds
0,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:05,2018-01-16 06:01:05,0.0
1,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:07,2018-01-16 06:01:07,0.0
2,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:08,2018-01-16 06:01:08,0.0
3,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:09,2018-01-16 06:01:09,0.0
4,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:44,2018-01-16 06:03:44,0.0


,zero_duration_episodes
app_name,
Google Chrome,11776
Facebook Messenger,5075
Minesweeper Classic (Mines),2873
Slidejoy,2752
Instagram,2255
Samsung Pay,2250
DigiHUD Pro Speedometer,2017
Facebook,1844
Google,1752


## 3. Build one row per session

Group records by both `user_id` and `session_id`. Use all event types so interaction-only sessions stay in the analysis. Three time measures answer different questions.

| Measure | What it captures |
|---|---|
| `event_span_seconds` | Time between the earliest and latest event timestamps. A single event has a span of zero. |
| `recorded_episode_seconds` | Sum of recorded episode durations. A session containing only interactions has a missing value. |
| `session_span_seconds` | Time from the earliest observed start to the latest observed end, using the event timestamp for point observations. |

Count short **episodes** here to establish a reference. Later we will compare this count with the number of short **bouts** after joining contiguous fragments.

In [3]:
events = df.assign(
    is_episode=is_episode,
    is_interaction=is_interaction,
    is_zero_episode=is_episode & df["duration_seconds"].eq(0),
    is_short_episode=is_episode & df["duration_seconds"].between(0, short_limit_seconds, inclusive="neither"),
    observed_start=df["start_time"].fillna(df["timestamp"]),
    observed_end=df["end_time"].fillna(df["timestamp"]),
)
sessions = events.groupby(session_keys, as_index=False).agg(
    first_event_time=("timestamp", "min"),
    last_event_time=("timestamp", "max"),
    event_count=("event_class", "size"),
    episode_count=("is_episode", "sum"),
    interaction_count=("is_interaction", "sum"),
    zero_duration_episode_count=("is_zero_episode", "sum"),
    short_episode_count=("is_short_episode", "sum"),
    session_start=("observed_start", "min"),
    session_end=("observed_end", "max"),
    unique_apps_observed=("app_name", "nunique"),
)
# Sum over episode rows only. The left join preserves missing duration for interaction-only sessions.
episode_seconds = df.loc[is_episode].groupby(session_keys)["duration_seconds"].sum()
sessions = sessions.join(episode_seconds.rename("recorded_episode_seconds"), on=session_keys, validate="one_to_one")
sessions["event_span_seconds"] = (sessions["last_event_time"] - sessions["first_event_time"]).dt.total_seconds()
sessions["session_span_seconds"] = (sessions["session_end"] - sessions["session_start"]).dt.total_seconds()

assert len(sessions) == n_sessions and sessions["event_count"].sum() == len(df)
assert (sessions["episode_count"] + sessions["interaction_count"]).equals(sessions["event_count"])
assert sessions["session_span_seconds"].ge(sessions["event_span_seconds"]).all()
assert sessions["event_span_seconds"].ge(0).all()
assert sessions["recorded_episode_seconds"].sum() == episode_seconds.sum()
assert sessions["short_episode_count"].sum() == events["is_short_episode"].sum()
assert sessions["short_episode_count"].le(sessions["episode_count"] - sessions["zero_duration_episode_count"]).all()

time_measures = ["event_span_seconds", "recorded_episode_seconds", "session_span_seconds"]
display(sessions[session_keys + ["event_count", "episode_count", "interaction_count"] + time_measures].head(10))

,user_id,session_id,event_count,episode_count,interaction_count,event_span_seconds,recorded_episode_seconds,session_span_seconds
0,0,1,31,31,0,192.0,0.0,192.0
1,0,2,4,3,1,27.0,21.0,32.0
2,0,3,1,1,0,0.0,51.0,51.0
3,0,4,36,36,0,348.0,1.0,348.0
4,0,5,39,39,0,126.0,0.0,126.0
5,0,6,41,41,0,210.0,1.0,210.0
6,0,7,40,40,0,354.0,0.0,354.0
7,0,8,2,0,2,0.0,NaN,0.0
8,0,9,1,0,1,0.0,NaN,0.0
9,0,10,32,32,0,119.0,1.0,119.0


### Which sessions have no positive episode duration?

A missing episode total identifies an interaction-only session. A total of zero means episodes were recorded, but none contributed positive duration. Counting both groups shows how much data a duration-only analysis would discard. The interaction-only event span can still be positive when two point observations occur at different times.

In [4]:
interaction_only = sessions["episode_count"].eq(0)
has_zero = sessions["zero_duration_episode_count"].gt(0)
has_positive = sessions["recorded_episode_seconds"].gt(0)
has_interaction = sessions["interaction_count"].gt(0)
assert sessions["recorded_episode_seconds"].isna().equals(interaction_only)

display(pd.Series({
    "Sessions with any zero-duration episode": has_zero.sum(),
    "Sessions with any positive episode duration": has_positive.sum(),
    "Sessions without a positive episode": (~has_positive).sum(),
    "  Only zero-duration episodes": (has_zero & ~has_positive & ~has_interaction).sum(),
    "  Zero-duration episodes and interactions": (has_zero & ~has_positive & has_interaction).sum(),
    "  Only interactions": interaction_only.sum(),
    "Interaction-only sessions with positive event span": (interaction_only & sessions["event_span_seconds"].gt(0)).sum(),
    "Interaction-only sessions with zero event span": (interaction_only & sessions["event_span_seconds"].eq(0)).sum(),
}, name="sessions").to_frame())

,sessions
Sessions with any zero-duration episode,23506
Sessions with any positive episode duration,55793
Sessions without a positive episode,15186
Only zero-duration episodes,1919
Zero-duration episodes and interactions,985
Only interactions,12282
Interaction-only sessions with positive event span,4370
Interaction-only sessions with zero event span,7912


### Which apps appear, and how does the user move between them?

Count distinct apps across all events, then separately in episodes and interactions. For navigation, follow `session_event_number` within each session. A change from A to B is a **transition**. In the sequence A → B → A, the return to A is also a **revisit**. Repeated records from A alone add neither.

In [5]:
for mask, column in [(is_episode, "unique_episode_apps"), (is_interaction, "unique_interaction_apps")]:
    app_counts = df.loc[mask].groupby(session_keys)["app_name"].nunique()
    sessions = sessions.join(app_counts.rename(column), on=session_keys, validate="one_to_one")
    sessions[column] = sessions[column].fillna(0).astype(int)

event_order = session_keys + ["session_event_number"]
print("Duplicate within-session event numbers:", df.duplicated(event_order).sum())
ordered = df.sort_values(event_order, kind="stable")[event_order + ["app_name"]].copy()
previous_app = ordered.groupby(session_keys)["app_name"].shift()
ordered["transition"] = previous_app.notna() & ordered["app_name"].notna() & ordered["app_name"].ne(previous_app)
ordered["revisit"] = ordered["transition"] & ordered.duplicated(session_keys + ["app_name"])

navigation = ordered.groupby(session_keys).agg(
    observed_app_transition_count=("transition", "sum"),
    observed_app_revisit_count=("revisit", "sum"),
)
sessions = sessions.join(navigation, on=session_keys, validate="one_to_one")
diversity_cols = ["unique_apps_observed", "unique_episode_apps", "unique_interaction_apps"]
assert sessions["unique_apps_observed"].le(sessions["event_count"]).all()
for column, count in zip(diversity_cols[1:], ["episode_count", "interaction_count"]):
    assert sessions[column].le(sessions[[count, "unique_apps_observed"]].min(axis=1)).all()
assert sessions["observed_app_transition_count"].le(sessions["event_count"] - 1).all()
assert sessions["observed_app_revisit_count"].le(sessions["observed_app_transition_count"]).all()
assert sessions["observed_app_transition_count"].sum() == ordered["transition"].sum()
assert sessions["observed_app_revisit_count"].sum() == ordered["revisit"].sum()
print("Sessions where all-event and episode-only app counts differ:",
      sessions["unique_apps_observed"].ne(sessions["unique_episode_apps"]).sum())
print("Observed transitions and revisits:", ordered[["transition", "revisit"]].sum().to_dict())
display(sessions[diversity_cols + list(navigation.columns)].describe().round(2))

Duplicate within-session event numbers: 0
Sessions where all-event and episode-only app counts differ: 31812
Observed transitions and revisits: {'transition': 234479, 'revisit': 135431}


,unique_apps_observed,unique_episode_apps,unique_interaction_apps,observed_app_transition_count,observed_app_revisit_count
count,70979.00,70979.00,70979.00,70979.0,70979.00
mean,2.40,1.82,0.97,3.3,1.91
std,1.68,1.58,1.08,6.8,5.68
min,1.00,0.00,0.00,0.0,0.00
25%,1.00,1.00,0.00,0.0,0.00
50%,2.00,1.00,1.00,1.0,0.00
75%,3.00,3.00,1.00,4.0,2.00
max,17.00,16.00,12.00,218.0,209.00


## 4. Work through one session with many short episodes

A single stretch of app use can appear as several consecutive episodes. Rank sessions by their short-episode count and inspect the highest one. In the supplied data this selects user 204, session 51034, the example used in the original analysis. Its event sequence lets us test a merge rule before applying it to everyone.

In [6]:
example_columns = session_keys + [
    "episode_count", "zero_duration_episode_count", "short_episode_count",
    "recorded_episode_seconds", "session_span_seconds", "observed_app_transition_count",
]
top_short_sessions = sessions.nlargest(10, "short_episode_count")
display(top_short_sessions[example_columns])

example_key = top_short_sessions.iloc[0][session_keys]
sample_events = df.loc[
    df["user_id"].eq(example_key["user_id"]) & df["session_id"].eq(example_key["session_id"])
].copy()
print(f"Worked session: user {example_key['user_id']}, session {example_key['session_id']}")

,user_id,session_id,episode_count,zero_duration_episode_count,short_episode_count,recorded_episode_seconds,session_span_seconds,observed_app_transition_count
47179,204,51034,1050,9,1014,3432.0,3622.0,20
1307,5,1559,946,12,883,5657.0,6014.0,43
1645,5,1928,940,14,830,7186.0,7496.0,26
65025,275,70059,935,7,805,7048.0,7261.0,28
1637,5,1918,971,31,754,11975.0,13029.0,95
48216,205,52164,1004,53,646,13531.0,15278.0,6
1669,5,1952,611,5,555,3611.0,3655.0,1
13211,68,14621,605,6,521,7672.0,8100.0,25
19313,112,21146,517,20,461,2711.0,3004.0,12
1294,5,1543,568,10,457,6256.0,6706.0,21


Worked session: user 204, session 51034


### Decide when two fragments belong to the same bout

Order episodes by start time, breaking ties with the recorded event number. Merge an episode with its predecessor only when **the app is the same**, **the two intervals touch exactly**, and **only one app appears among all events at that boundary timestamp**. Any failed test starts a new bout. The first episode in every session always starts a bout.

The function returns both the fragment-level decisions and the reconstructed intervals. Keeping both makes it possible to inspect each merge and to check that total recorded duration is preserved.

In [7]:
def reconstruct_bouts(records):
    """Return ordered episode fragments and the bouts formed within each session."""
    boundary_apps = records.groupby(session_keys + ["timestamp"])["app_name"].nunique()
    fields = session_keys + ["session_event_number", "app_name", "start_time", "end_time", "duration_seconds"]
    fragments = records.loc[records["event_class"].eq("episode"), fields].copy()
    fragments = fragments.join(
        boundary_apps.rename("apps_at_boundary"),
        on=session_keys + ["start_time"], validate="many_to_one",
    ).sort_values(session_keys + ["start_time", "session_event_number"], kind="stable").reset_index(drop=True)

    within_session = fragments.groupby(session_keys)
    fragments["same_app"] = fragments["app_name"].eq(within_session["app_name"].shift())
    previous_end = within_session["end_time"].shift()
    fragments["gap_seconds"] = (fragments["start_time"] - previous_end).dt.total_seconds()
    fragments["merge_with_previous"] = (
        fragments["same_app"] & fragments["gap_seconds"].eq(0) & fragments["apps_at_boundary"].eq(1)
    )
    # Cumulative starts give every fragment a bout number local to its session.
    fragments["new_bout"] = ~fragments["merge_with_previous"]
    fragments["bout_id"] = fragments.groupby(session_keys)["new_bout"].cumsum()
    bouts = fragments.groupby(session_keys + ["bout_id"], as_index=False).agg(
        app_name=("app_name", "first"), start_time=("start_time", "min"),
        end_time=("end_time", "max"), fragment_count=("app_name", "size"),
    )
    bouts["duration_seconds"] = (bouts["end_time"] - bouts["start_time"]).dt.total_seconds()
    bouts["is_short_bout"] = bouts["duration_seconds"].between(0, short_limit_seconds, inclusive="neither")

    assert bouts["fragment_count"].sum() == len(fragments)
    assert len(bouts) == len(fragments) - fragments["merge_with_previous"].sum()
    assert np.isclose(bouts["duration_seconds"].sum(), fragments["duration_seconds"].sum())
    return fragments, bouts

In [8]:
sample_fragments, sample_bouts = reconstruct_bouts(sample_events)
same_app_gaps = sample_fragments.loc[sample_fragments["same_app"], "gap_seconds"]
touching_pairs = sample_fragments["same_app"] & sample_fragments["gap_seconds"].eq(0)

display(pd.Series({
    "Episode fragments": len(sample_fragments),
    "Consecutive same-app pairs": len(same_app_gaps),
    "Pairs touching / separated / overlapping": (same_app_gaps.eq(0).sum(), same_app_gaps.gt(0).sum(), same_app_gaps.lt(0).sum()),
    "Touching pairs with another app at the boundary": (touching_pairs & sample_fragments["apps_at_boundary"].gt(1)).sum(),
    "Accepted merges": sample_fragments["merge_with_previous"].sum(),
    "Reconstructed bouts": len(sample_bouts),
    "Short bouts": sample_bouts["is_short_bout"].sum(),
    "Recorded seconds before / after": (sample_fragments["duration_seconds"].sum(), sample_bouts["duration_seconds"].sum()),
}, name="worked session").to_frame())
display(sample_fragments["app_name"].value_counts().head(10).rename("episode_count").to_frame())
display(pd.concat([sample_fragments["duration_seconds"].describe(), same_app_gaps.describe()], axis=1).round(2))
display(sample_fragments[["app_name", "start_time", "end_time", "gap_seconds", "apps_at_boundary",
                          "merge_with_previous", "bout_id"]].head(20))

,worked session
Episode fragments,1050
Consecutive same-app pairs,1033
Pairs touching / separated / overlapping,"(979, 54, 0)"
Touching pairs with another app at the boundary,1
Accepted merges,978
Reconstructed bouts,72
Short bouts,26
Recorded seconds before / after,"(3432.0, 3432.0)"


,episode_count
app_name,
Settings,982
YouTube,44
Gmail,22
Verizon Messages,1
Google,1


,duration_seconds,gap_seconds
count,1050.00,1033.00
mean,3.27,0.18
std,11.14,3.35
min,0.00,0.00
25%,1.00,0.00
50%,2.00,0.00
75%,3.00,0.00
max,279.00,107.00


,app_name,start_time,end_time,gap_seconds,apps_at_boundary,merge_with_previous,bout_id
0,YouTube,2018-04-18 15:03:58,2018-04-18 15:04:03,NaN,1,False,1
1,YouTube,2018-04-18 15:04:03,2018-04-18 15:04:04,0.0,1,True,1
2,YouTube,2018-04-18 15:04:04,2018-04-18 15:04:09,0.0,1,True,1
3,YouTube,2018-04-18 15:04:09,2018-04-18 15:04:16,0.0,1,True,1
4,YouTube,2018-04-18 15:04:16,2018-04-18 15:05:12,0.0,1,True,1
5,YouTube,2018-04-18 15:05:12,2018-04-18 15:05:14,0.0,1,True,1
6,YouTube,2018-04-18 15:05:14,2018-04-18 15:05:17,0.0,1,True,1
7,YouTube,2018-04-18 15:05:17,2018-04-18 15:05:19,0.0,1,True,1
8,YouTube,2018-04-18 15:05:19,2018-04-18 15:05:22,0.0,1,True,1
9,YouTube,2018-04-18 15:05:22,2018-04-18 15:05:23,0.0,1,True,1


### Inspect activity inside the example bouts

A clear shared boundary does not guarantee uninterrupted use. For each example bout, inspect the original records from other apps. The pilot check looks for an interaction strictly inside the bout or an episode that meets the overlap inequalities below. The output identifies the conflicting merged bouts and shows the records behind each flag. Single-fragment bouts are included in the overall check.

In [9]:
def conflicting_records(bout, records):
    other = records.loc[records["app_name"].ne(bout.app_name)]
    inside = other["event_class"].eq("interaction") & other["timestamp"].between(
        bout.start_time, bout.end_time, inclusive="neither"
    )
    overlap = (other["event_class"].eq("episode")
               & other["start_time"].lt(bout.end_time) & other["end_time"].gt(bout.start_time))
    return other.loc[inside | overlap]

sample_bouts["has_conflict"] = [
    not conflicting_records(bout, sample_events).empty for bout in sample_bouts.itertuples()
]
merged = sample_bouts["fragment_count"].gt(1)
flagged = sample_bouts["has_conflict"]
short = sample_bouts["is_short_bout"]
display(pd.Series({
    "Bouts with multiple fragments": merged.sum(),
    "Merged bouts with a conflict": (merged & flagged).sum(),
    "All bouts with a conflict": flagged.sum(),
    "Short bouts with a conflict": (short & flagged).sum(),
    "Short bouts without a detected conflict": (short & ~flagged).sum(),
}, name="worked session").to_frame())

conflicted_merges = sample_bouts.loc[merged & flagged]
display(conflicted_merges[["bout_id", "app_name", "start_time", "end_time", "fragment_count", "duration_seconds"]])
for bout in conflicted_merges.itertuples():
    evidence = conflicting_records(bout, sample_events)
    print(f"Bout {bout.bout_id} ({bout.app_name}) — {len(evidence)} conflicting records")
    display(evidence[["app_name", "event_class", "timestamp", "start_time", "end_time"]].head(15))

,worked session
Bouts with multiple fragments,47
Merged bouts with a conflict,3
All bouts with a conflict,10
Short bouts with a conflict,8
Short bouts without a detected conflict,18


,bout_id,app_name,start_time,end_time,fragment_count,duration_seconds
34,35,Settings,2018-04-18 15:29:58,2018-04-18 15:30:41,18,43.0
60,61,Settings,2018-04-18 15:46:20,2018-04-18 15:46:23,3,3.0
65,66,Gmail,2018-04-18 15:46:28,2018-04-18 15:47:53,18,85.0


Bout 35 (Settings) — 3 conflicting records


,app_name,event_class,timestamp,start_time,end_time
597259,Facebook Messenger,interaction,2018-04-18 15:30:30,NaT,NaT
597260,Facebook Messenger,interaction,2018-04-18 15:30:30,NaT,NaT
597261,Facebook Messenger,interaction,2018-04-18 15:30:35,NaT,NaT


Bout 61 (Settings) — 1 conflicting records


,app_name,event_class,timestamp,start_time,end_time
597584,Gmail,episode,2018-04-18 15:46:20,2018-04-18 15:46:20,2018-04-18 15:46:23


Bout 66 (Gmail) — 1 conflicting records


,app_name,event_class,timestamp,start_time,end_time
597591,Settings,episode,2018-04-18 15:46:26,2018-04-18 15:46:26,2018-04-18 15:46:30


## 5. Reconstruct and audit bouts across the dataset

Apply the same boundary rule to every session. Compare the number of use units before and after merging, for all episodes and for short episodes only. The percentage reduction describes how sensitive each count is to fragmentation. The duration check inside `reconstruct_bouts` ensures that joining touching intervals preserves recorded seconds.

In [10]:
all_fragments, all_bouts = reconstruct_bouts(df)
before = pd.Series({"All use units": len(all_fragments), "Short use units": sessions["short_episode_count"].sum()})
after = pd.Series({"All use units": len(all_bouts), "Short use units": all_bouts["is_short_bout"].sum()})
unit_comparison = pd.DataFrame({"Episodes": before, "Bouts": after})
unit_comparison["Reduction (%)"] = ((1 - after / before) * 100).round(2)
display(unit_comparison)
print("Accepted merges:", all_fragments["merge_with_previous"].sum())
print("Unmatched boundary timestamps:", all_fragments["apps_at_boundary"].isna().sum())
print("Recorded seconds before and after:", all_fragments["duration_seconds"].sum(), all_bouts["duration_seconds"].sum())

,Episodes,Bouts,Reduction (%)
All use units,585520,334613,42.85
Short use units,346944,131808,62.01


Accepted merges: 250907
Unmatched boundary timestamps: 0
Recorded seconds before and after: 14161098.0 14161098.0


### Flag overlapping app intervals

The dataset-wide audit tests **positive-duration bouts** within the same session. Two different apps conflict when each starts before the other ends. Strict inequalities allow two bouts to touch at a boundary. Each row of the comparison matrix represents one bout, so a single overlap flags both participants.

In [11]:
bout_audit = all_bouts.reset_index(drop=True).copy()
starts = bout_audit["start_time"].to_numpy(dtype="datetime64[ns]")
ends = bout_audit["end_time"].to_numpy(dtype="datetime64[ns]")
apps = bout_audit["app_name"].to_numpy()
cross_app_overlap = np.zeros(len(bout_audit), dtype=bool)

# Form a separate comparison matrix per session so intervals from different sessions never meet.
for positions in bout_audit.groupby(session_keys, sort=False).indices.values():
    positive = positions[ends[positions] > starts[positions]]
    if len(positive) < 2 or len(np.unique(apps[positive])) < 2:
        continue
    start, end, app = starts[positive], ends[positive], apps[positive]
    overlaps = ((start[:, None] < end[None, :]) & (end[:, None] > start[None, :])
                & (app[:, None] != app[None, :]))
    cross_app_overlap[positive] = overlaps.any(axis=1)

bout_audit["cross_app_overlap"] = cross_app_overlap
print("Bouts with cross-app overlap:", cross_app_overlap.sum())
print("Sessions containing an overlap:", bout_audit.loc[cross_app_overlap, session_keys].drop_duplicates().shape[0])
print("Short bouts with an overlap:", (bout_audit["is_short_bout"] & cross_app_overlap).sum())

Bouts with cross-app overlap: 13497
Sessions containing an overlap: 2758
Short bouts with an overlap: 8688


### Check for other-app interactions inside each bout

For each interval, count interactions from **all apps in the session**, then count interactions from **the bout's own app**. A higher first count means another app was active inside. Interactions at the exact start or end are excluded. A bout is flagged if either this test or the interval-overlap test finds a conflict. Only unflagged short bouts contribute to subsequent frequency estimates.

In [12]:
interactions = df.loc[is_interaction, session_keys + ["app_name", "timestamp"]].dropna()
interactions = interactions.sort_values("timestamp", kind="stable").reset_index(drop=True)

def count_inside_bouts(bouts, interactions, group_keys):
    """Count interactions strictly between each bout's endpoints within matching groups."""
    time_groups = interactions.groupby(group_keys, sort=False).indices
    times = interactions["timestamp"].to_numpy(dtype="datetime64[ns]")
    starts = bouts["start_time"].to_numpy(dtype="datetime64[ns]")
    ends = bouts["end_time"].to_numpy(dtype="datetime64[ns]")
    counts = np.zeros(len(bouts), dtype=int)

    for key, positions in bouts.groupby(group_keys, sort=False).indices.items():
        if key not in time_groups:
            continue
        # Records were sorted by timestamp, so the selected group's times are already ordered.
        observed = times[time_groups[key]]
        left = np.searchsorted(observed, starts[positions], side="right")
        right = np.searchsorted(observed, ends[positions], side="left")
        counts[positions] = np.maximum(right - left, 0)  # Empty intervals contain no interactions.
    return counts

total_inside = count_inside_bouts(bout_audit, interactions, session_keys)
same_app_inside = count_inside_bouts(bout_audit, interactions, session_keys + ["app_name"])
bout_audit["other_app_interaction_inside"] = total_inside > same_app_inside
bout_audit["any_detected_conflict"] = bout_audit["cross_app_overlap"] | bout_audit["other_app_interaction_inside"]
bout_audit["is_short_bout_unflagged"] = bout_audit["is_short_bout"] & ~bout_audit["any_detected_conflict"]
bout_audit["is_short_bout_conflicted"] = bout_audit["is_short_bout"] & bout_audit["any_detected_conflict"]
flags = ["other_app_interaction_inside", "any_detected_conflict", "is_short_bout_conflicted", "is_short_bout_unflagged"]
display(bout_audit[flags].sum().rename("bouts").to_frame())

,bouts
other_app_interaction_inside,20161
any_detected_conflict,32305
is_short_bout_conflicted,12959
is_short_bout_unflagged,118849


### Bring the audited counts back to sessions

A session with no episodes contributes zero bouts. Within every other session, the flagged and unflagged short counts must add back to the total short-bout count. Compare the session-level distributions to see where fragment merging changes the short-use measure.

In [13]:
bout_counts = bout_audit.groupby(session_keys).agg(
    bout_count=("bout_id", "size"), short_bout_count=("is_short_bout", "sum"),
    conflicted_bout_count=("any_detected_conflict", "sum"),
    short_bout_unflagged_count=("is_short_bout_unflagged", "sum"),
    short_bout_conflicted_count=("is_short_bout_conflicted", "sum"),
)
sessions = sessions.join(bout_counts, on=session_keys, validate="one_to_one")
sessions[bout_counts.columns] = sessions[bout_counts.columns].fillna(0).astype(int)

assert len(sessions) == n_sessions and sessions["bout_count"].sum() == len(bout_audit)
assert sessions["bout_count"].le(sessions["episode_count"]).all()
assert sessions["short_bout_count"].sum() == bout_audit["is_short_bout"].sum()
assert sessions["short_bout_unflagged_count"].sum() == bout_audit["is_short_bout_unflagged"].sum()
assert sessions["short_bout_count"].eq(sessions["short_bout_unflagged_count"] + sessions["short_bout_conflicted_count"]).all()
print("Sessions containing episodes:", sessions["episode_count"].gt(0).sum())
print("Sessions with fewer units after merging:", sessions["bout_count"].lt(sessions["episode_count"]).sum())
print("Sessions with fewer short units:", sessions["short_bout_count"].lt(sessions["short_episode_count"]).sum())
display(sessions[["episode_count", "short_episode_count"] + list(bout_counts.columns)].describe(
    percentiles=[.5, .75, .9, .95, .99]
).round(2))

Sessions containing episodes: 58697
Sessions with fewer units after merging: 28132
Sessions with fewer short units: 27047


,episode_count,short_episode_count,bout_count,short_bout_count,conflicted_bout_count,short_bout_unflagged_count,short_bout_conflicted_count
count,70979.00,70979.00,70979.00,70979.00,70979.00,70979.00,70979.00
mean,8.25,4.89,4.71,1.86,0.46,1.67,0.18
std,20.79,15.67,8.33,3.90,2.07,3.21,1.25
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,3.00,1.00,2.00,1.00,0.00,1.00,0.00
75%,8.00,4.00,5.00,2.00,0.00,2.00,0.00
90%,19.00,11.00,11.00,5.00,1.00,4.00,0.00
95%,32.00,19.00,17.00,7.00,2.00,7.00,1.00
99%,81.00,54.00,39.00,17.00,7.00,15.00,4.00
max,1050.00,1014.00,245.00,186.00,167.00,97.00,107.00


## 6. Measure recent activity at each session start

The decision point is the start of the current session. First sort each user's sessions chronologically and measure the gap from the previous session's end. The first observed session for each user has no preceding gap.

Count session starts in 30- and 60-minute windows, including the lower time boundary. The rolling count includes the current row, so subtract one to obtain the number of prior starts. A session exactly 30 minutes earlier contributes to both windows.

In [14]:
sessions = sessions.sort_values(order_columns).reset_index(drop=True)
sessions["previous_session_end"] = sessions.groupby("user_id")["session_end"].shift()
gap = (sessions["session_start"] - sessions["previous_session_end"]).dt.total_seconds()
sessions["inter_session_gap_seconds"] = gap
assert gap.isna().equals(sessions.groupby("user_id").cumcount().eq(0))
print(f"Gaps: {gap.isna().sum():,} missing, {gap.eq(0).sum():,} zero, "
      f"{gap.gt(0).sum():,} positive, {gap.lt(0).sum():,} negative")
display(gap.describe().rename("inter_session_gap_seconds").to_frame())

def rolling_starts(timestamps, minutes):
    markers = pd.Series(1, index=pd.DatetimeIndex(timestamps))
    return markers.rolling(f"{minutes}min", closed="both").sum().to_numpy()

for minutes in (30, 60):
    total = f"sessions_last_{minutes}min"
    sessions[total] = sessions.groupby("user_id")["session_start"].transform(rolling_starts, minutes).astype(int)
    sessions[f"prior_sessions_{minutes}min"] = sessions[total] - 1

assert sessions["prior_sessions_30min"].between(0, sessions["prior_sessions_60min"]).all()
display(sessions[["sessions_last_30min", "sessions_last_60min"]].describe().round(2))

Gaps: 291 missing, 0 zero, 70,688 positive, 0 negative


,inter_session_gap_seconds
count,7.068800e+04
mean,5.247838e+03
std,1.009529e+05
min,3.010000e+02
25%,4.810000e+02
50%,8.200000e+02
75%,1.757000e+03
max,9.698402e+06


,sessions_last_30min,sessions_last_60min
count,70979.00,70979.00
mean,1.92,2.99
std,0.80,1.29
min,1.00,1.00
25%,1.00,2.00
50%,2.00,3.00
75%,2.00,4.00
max,5.00,9.00


### Count completed short bouts in the same windows

For a session starting at time *t*, count this user's unflagged short bouts with end times in **[t − window, t)**. A bout ending exactly at *t* is excluded. This uses only completed activity and keeps the features available at the decision point. These counts measure short app use as defined by the log. The fields do not establish whether a bout was a glance in the sense used by Parry and Toth.

In [15]:
eligible_bouts = bout_audit.loc[bout_audit["is_short_bout_unflagged"]]
bout_ends_by_user = {
    user: np.sort(rows["end_time"].to_numpy(dtype="datetime64[ns]"))
    for user, rows in eligible_bouts.groupby("user_id")
}
for minutes in (30, 60):
    sessions[f"prior_short_bouts_{minutes}min"] = 0

for user, indices in sessions.groupby("user_id").groups.items():
    if user not in bout_ends_by_user:
        continue
    ends = bout_ends_by_user[user]
    starts = sessions.loc[indices, "session_start"].to_numpy(dtype="datetime64[ns]")
    # Both left insertions give an inclusive lower boundary and an exclusive upper boundary.
    before_start = np.searchsorted(ends, starts, side="left")
    for minutes in (30, 60):
        lower = np.searchsorted(ends, starts - np.timedelta64(minutes, "m"), side="left")
        sessions.loc[indices, f"prior_short_bouts_{minutes}min"] = before_start - lower

assert sessions["prior_short_bouts_30min"].between(0, sessions["prior_short_bouts_60min"]).all()
assert sessions.groupby("user_id").head(1)["prior_short_bouts_60min"].eq(0).all()
display(sessions[["prior_short_bouts_30min", "prior_short_bouts_60min"]].describe(
    percentiles=[.5, .75, .9, .95, .99]
).round(2))

,prior_short_bouts_30min,prior_short_bouts_60min
count,70979.00,70979.00
mean,1.54,3.37
std,2.62,4.73
min,0.00,0.00
50%,0.00,2.00
75%,2.00,5.00
90%,4.00,9.00
95%,6.00,12.00
99%,12.00,21.00
max,74.00,97.00


### How closely do the two frequencies track each other?

One prior session can contain several short bouts. Compare the two 60-minute counts using Spearman's rank correlation, then inspect how widely the short-bout counts vary among sessions with the same number of prior starts.

In [16]:
recent_counts = ["prior_sessions_60min", "prior_short_bouts_60min"]
assert sessions[recent_counts].notna().all().all()
# Spearman correlation is the Pearson correlation of the two average-rank series.
rank_correlation = sessions[recent_counts].rank(method="average").corr()
print(f"Spearman correlation: {rank_correlation.iloc[0, 1]:.3f}")

frequency_comparison = sessions.groupby("prior_sessions_60min")["prior_short_bouts_60min"].agg(
    session_count="size", median_short_bouts="median", mean_short_bouts="mean",
    p90_short_bouts=lambda counts: counts.quantile(.90), max_short_bouts="max",
)
display(frequency_comparison.round(2))

Spearman correlation: 0.450


,session_count,median_short_bouts,mean_short_bouts,p90_short_bouts,max_short_bouts
prior_sessions_60min,,,,,
0,10089,0.0,0.59,0.0,85
1,16221,1.0,2.69,7.0,97
2,20068,2.0,3.91,9.0,86
3,15769,3.0,4.41,10.0,90
4,6981,4.0,4.64,10.0,45
5,1639,4.0,4.80,10.0,37
6,197,3.0,4.42,10.0,23
7,12,3.0,5.75,17.3,18
8,3,21.0,20.67,21.0,21


### Assemble the decision-point table

Copy the four prior-window counts and the previous-session gap into `decision_features`. Add the recorded hour, day of week (Monday = 0), and a marker for each user's first session. Measures that depend on how the current session ends stay in `sessions`.

In [17]:
history_counts = ["prior_sessions_30min", "prior_sessions_60min", "prior_short_bouts_30min", "prior_short_bouts_60min"]
decision_columns = session_keys + ["session_start", "inter_session_gap_seconds"] + history_counts
decision_features = sessions[decision_columns].copy()
decision_features["hour_of_day"] = decision_features["session_start"].dt.hour
decision_features["day_of_week"] = decision_features["session_start"].dt.dayofweek
decision_features["first_observed_session"] = decision_features["inter_session_gap_seconds"].isna()

assert len(decision_features) == n_sessions and not decision_features.duplicated(session_keys).any()
assert decision_features["hour_of_day"].between(0, 23).all() and decision_features["day_of_week"].between(0, 6).all()
assert decision_features["first_observed_session"].sum() == n_users
display(decision_features.head(10))
display(decision_features.isna().sum().rename("missing_values").to_frame())
for column in ["hour_of_day", "day_of_week"]:
    display(decision_features[column].value_counts().sort_index().rename("session_starts").to_frame())

,user_id,session_id,session_start,inter_session_gap_seconds,prior_sessions_30min,prior_sessions_60min,prior_short_bouts_30min,prior_short_bouts_60min,hour_of_day,day_of_week,first_observed_session
0,0,1,2018-01-16 06:01:05,NaN,0,0,0,0,6,1,True
1,0,2,2018-01-16 06:25:54,1297.0,1,1,0,0,6,1,False
2,0,3,2018-01-16 06:35:35,549.0,1,2,3,3,6,1,False
3,0,4,2018-01-16 07:15:56,2370.0,0,2,0,3,7,1,False
4,0,5,2018-01-16 08:02:05,2421.0,0,1,0,1,8,1,False
5,0,6,2018-01-16 08:30:44,1593.0,1,1,0,0,8,1,False
6,0,7,2018-01-16 09:45:51,4297.0,0,0,0,0,9,1,False
7,0,8,2018-01-16 09:59:57,492.0,1,1,0,0,9,1,False
8,0,9,2018-01-16 10:29:54,1797.0,1,2,0,0,10,1,False
9,0,10,2018-01-16 11:01:28,1894.0,0,1,0,0,11,1,False


,missing_values
user_id,0
session_id,0
session_start,0
inter_session_gap_seconds,291
prior_sessions_30min,0
prior_sessions_60min,0
prior_short_bouts_30min,0
prior_short_bouts_60min,0
hour_of_day,0
day_of_week,0


,session_starts
hour_of_day,
0,3561
1,3275
2,3065
3,2681
4,2313
5,2047
6,1633
7,1526
8,1408


,session_starts
day_of_week,
0,10701
1,10551
2,10326
3,9882
4,9638
5,9996
6,9885


## 7. Estimate each user's historical baseline

For each 60-minute count, calculate that user's expanding 75th and 90th percentiles (P75 and P90). Shift the observations by one session **before** computing the percentiles so the current count cannot move its own threshold. A baseline first becomes available after 10 previous session starts.

A count is **elevated** only when it strictly exceeds the matching percentile. A tie is below the elevation criterion. Leave the indicator missing during warm-up so an unknown baseline stays distinct from an observed low count.

In [18]:
decision_features = decision_features.sort_values(order_columns).reset_index(drop=True)
by_user = decision_features.groupby("user_id")
decision_features["history_session_count"] = by_user.cumcount()
has_baseline = decision_features["history_session_count"].ge(10)
metrics = [
    ("prior_sessions_60min", "baseline_sessions", "elevated_session_frequency"),
    ("prior_short_bouts_60min", "baseline_short_bouts", "elevated_short_bout_frequency"),
]
baseline_cols = []
for percentile in (75, 90):
    for value_col, baseline_prefix, flag_prefix in metrics:
        baseline_col = f"{baseline_prefix}_p{percentile}"
        flag_col = flag_prefix if percentile == 75 else f"{flag_prefix}_p90"
        baseline_cols.append(baseline_col)
        decision_features[baseline_col] = by_user[value_col].transform(
            lambda history: history.shift(1).expanding(min_periods=10).quantile(percentile / 100)
        )
        decision_features[flag_col] = (
            decision_features[value_col].gt(decision_features[baseline_col]).astype("boolean").where(has_baseline)
        )
        assert decision_features[baseline_col].notna().equals(has_baseline)
        assert decision_features[flag_col].notna().equals(has_baseline)

print(f"Session starts with a personal baseline: {has_baseline.sum():,}")
display(decision_features[baseline_cols].describe().round(2))

Session starts with a personal baseline: 68,147


,baseline_sessions_p75,baseline_short_bouts_p75,baseline_sessions_p90,baseline_short_bouts_p90
count,68147.00,68147.00,68147.00,68147.00
mean,2.75,4.87,3.51,8.11
std,0.54,2.91,0.65,4.47
min,0.00,0.00,0.00,0.00
25%,2.25,3.00,3.00,5.00
50%,3.00,4.00,4.00,7.00
75%,3.00,6.00,4.00,10.20
max,7.25,39.50,8.00,64.00


### Compare P75 and P90 on the same observations

Each cross-tabulation shows whether prior session starts, prior short bouts, both, or neither exceed their respective thresholds. Percentages use only sessions that have at least 10 earlier observations. Raising the threshold shows how sensitive the elevation rates are to the definition.

In [19]:
for percentile, suffix in [(75, ""), (90, "_p90")]:
    session_flag = "elevated_session_frequency" + suffix
    bout_flag = "elevated_short_bout_frequency" + suffix
    flags = decision_features.loc[has_baseline, [session_flag, bout_flag]]
    print(f"P{percentile} — elevated session frequency: {flags[session_flag].mean():.2%}, "
          f"elevated short-bout frequency: {flags[bout_flag].mean():.2%}")
    display(pd.crosstab(flags[session_flag], flags[bout_flag], margins=True,
                        rownames=["Elevated session frequency"], colnames=["Elevated short-bout frequency"]))

P75 — elevated session frequency: 16.99%, elevated short-bout frequency: 22.50%


Elevated short-bout frequency,False,True,All
Elevated session frequency,,,
False,45345,11226,56571
True,7466,4110,11576
All,52811,15336,68147


P90 — elevated session frequency: 6.45%, elevated short-bout frequency: 9.93%


Elevated short-bout frequency,False,True,All
Elevated session frequency,,,
False,57776,5976,63752
True,3606,789,4395
All,61382,6765,68147


## 8. Assess the warm-up period

A longer history gives each personal baseline more observations, while leaving fewer session starts eligible for classification. Count the sessions and users available after 10, 20, 30 and 50 earlier observations.

Then compare each user's threshold at 10 observations with their threshold at 50. For the short-bout thresholds, also compare 20 and 30 with 50. Use users with every snapshot required for each comparison and summarise the absolute change in units of the underlying frequency count.

In [20]:
history = decision_features["history_session_count"]
coverage = []
for n in (10, 20, 30, 50):
    eligible = decision_features.loc[history.ge(n), "user_id"]
    coverage.append({"Earlier observations": n, "Eligible starts": len(eligible), "Users": eligible.nunique()})
display(pd.DataFrame(coverage).set_index("Earlier observations"))

def compare_snapshots(columns, stages):
    snapshots = decision_features.loc[history.isin(stages)].pivot(
        index="user_id", columns="history_session_count", values=columns
    )
    # Keep the same users throughout a given set of comparisons.
    required = pd.MultiIndex.from_product([columns, stages])
    snapshots = snapshots.reindex(columns=required).dropna()
    rows = []
    for column in columns:
        for earlier in stages[:-1]:
            later = stages[-1]
            change = (snapshots[(column, later)] - snapshots[(column, earlier)]).abs()
            rows.append({"Baseline": column, "Earlier": earlier, "Later": later, "Users": len(change),
                         "Median change": change.median(), "P90 change": change.quantile(.90),
                         "Maximum change": change.max()})
    return pd.DataFrame(rows).round(2)

display(compare_snapshots(baseline_cols, [10, 50]))
display(compare_snapshots(["baseline_short_bouts_p75", "baseline_short_bouts_p90"], [20, 30, 50]))

,Eligible starts,Users
Earlier observations,,
10,68147,271
20,65513,255
30,63054,240
50,58615,196


,Baseline,Earlier,Later,Users,Median change,P90 change,Maximum change
0,baseline_sessions_p75,10,50,196,0.75,1.25,3.0
1,baseline_short_bouts_p75,10,50,196,1.38,5.62,20.0
2,baseline_sessions_p90,10,50,196,0.90,1.90,3.0
3,baseline_short_bouts_p90,10,50,196,2.65,7.65,17.2


,Baseline,Earlier,Later,Users,Median change,P90 change,Maximum change
0,baseline_short_bouts_p75,20,50,196,1.0,4.25,12.75
1,baseline_short_bouts_p75,30,50,196,1.0,3.25,10.75
2,baseline_short_bouts_p90,20,50,196,1.8,6.10,13.30
3,baseline_short_bouts_p90,30,50,196,1.0,4.95,13.10


## 9. Assign the engagement pattern

The final rule requires **at least 30 earlier session starts**. A frequency is elevated when it is **strictly above the user's expanding P75**. The 30-observation requirement controls when a label is assigned. The baseline still uses all preceding observations. P90 serves as the sensitivity comparison.

| `engagement_pattern_p75` | Meaning at this session start |
|---|---|
| `both_elevated` | Both 60-minute counts exceed their respective personal P75. |
| `session_only` | Only the number of prior session starts exceeds P75. |
| `short_bout_only` | Only the number of completed, unflagged short bouts exceeds P75. |
| `neither_elevated` | Both counts are at or below their respective P75. |
| `insufficient_history` | Fewer than 30 earlier session starts are available. |

In [21]:
eligible_30 = history.ge(30)
# Warm-up rows receive their own label first, including rows with 10–29 earlier starts.
session_elevated = decision_features["prior_sessions_60min"].gt(decision_features["baseline_sessions_p75"])
bout_elevated = decision_features["prior_short_bouts_60min"].gt(decision_features["baseline_short_bouts_p75"])
decision_features["engagement_pattern_p75"] = np.select(
    [~eligible_30, session_elevated & bout_elevated, session_elevated, bout_elevated],
    ["insufficient_history", "both_elevated", "session_only", "short_bout_only"],
    default="neither_elevated",
)

patterns = decision_features["engagement_pattern_p75"]
assert patterns.eq("insufficient_history").equals(~eligible_30)
assert patterns.isin(["session_only", "both_elevated"]).equals(eligible_30 & session_elevated)
assert patterns.isin(["short_bout_only", "both_elevated"]).equals(eligible_30 & bout_elevated)
counts = patterns.value_counts().rename("Session starts")
assert counts.sum() == n_sessions
distribution = counts.to_frame()
distribution["Share of all starts (%)"] = (counts / n_sessions * 100).round(2)
display(distribution)

,Session starts,Share of all starts (%)
engagement_pattern_p75,,
neither_elevated,42372,59.70
short_bout_only,10324,14.55
insufficient_history,7925,11.17
session_only,6849,9.65
both_elevated,3509,4.94


### Read a label back against its thresholds

The distribution above uses **all** session starts as the denominator, including those still in warm-up. The rows below give one example of each observed pattern. Read the two recent counts alongside their personal thresholds and check the amount of history first. The label describes the user's recent activity at that moment. A `short_bout_only` label, for example, means that brief app-use bouts were unusually frequent for this user even though session starts were within their usual range.

In [22]:
label_columns = session_keys + [
    "session_start", "history_session_count",
    "prior_sessions_60min", "baseline_sessions_p75",
    "prior_short_bouts_60min", "baseline_short_bouts_p75", "engagement_pattern_p75",
]
examples = decision_features.drop_duplicates("engagement_pattern_p75")
display(examples[label_columns].sort_values("engagement_pattern_p75").reset_index(drop=True))

,user_id,session_id,session_start,history_session_count,prior_sessions_60min,baseline_sessions_p75,prior_short_bouts_60min,baseline_short_bouts_p75,engagement_pattern_p75
0,0,31,2018-01-16 22:25:53,30,3,2.0,4,2.00,both_elevated
1,0,1,2018-01-16 06:01:05,0,0,NaN,0,NaN,insufficient_history
2,0,35,2018-01-17 01:30:39,34,0,2.0,0,3.00,neither_elevated
3,0,38,2018-01-17 02:14:05,37,3,2.0,0,3.00,session_only
4,0,33,2018-01-16 23:21:38,32,2,2.0,11,2.25,short_bout_only


### What the supplied LSApp workbook shows

Of the 70,979 session starts, 63,054 meet the 30-session history requirement, covering 240 users. Both frequencies are elevated at 3,509 starts. At another 10,324 starts, short-bout frequency alone is elevated. Counting session starts alone would therefore miss a distinct pattern of brief app use.

The unit of measurement matters for that result. Merging touching same-app fragments reduces 346,944 short episodes to 131,808 short bouts while preserving total recorded duration. Of those bouts, 118,849 pass both conflict checks and supply the history used for classification.